# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaminari19/FlyRank-Starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


The lane of my choice is the Refresh / Content Opportunity Scoring. I'm picking this lane because it maps directly onto a decision someone already has to make every week (which pages does a content editor look at first?), and because the starter pipeline already shows a plain rule leaves real value on the table — the baseline rule scores precision@50 = 0.240 while the random forest scores 0.740 on the same slice (`outputs/model_report.md`). That gap is evidence that a learned ranking beats a fixed rule *on this starter slice*, which is exactly the kind of gap worth spending 7 weeks investigating properly (with a future-window label instead of the beginner proxy, and honest validation). Lane 2 also has the deepest, most direct support in both the starter CSV and the warehouse (`dim_content` + `fact_content_daily_performance`), so I'm not betting the project on a sparse signal.

In [18]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kaminari19/FlyRank-Starter"
REPO_DIR = "FlyRank-Starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started (handles work/notebooks/ or repo root)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")

report = open("outputs/model_report.md").read()

rows = {}
for line in report.splitlines():
    line = line.strip()
    if line.startswith("| baseline_rules") or line.startswith("| random_forest"):
        cols = [c.strip() for c in line.strip("|").split("|")]
        model, precision_at_50 = cols[0], cols[3]
        rows[model] = float(precision_at_50)

base = rows["baseline_rules"]
rf = rows["random_forest"]

print(f"Hand-written rule  Precision@50: {base:.3f}   (~{round(base*50)} of the top 50 right)")
print(f"Random forest      Precision@50: {rf:.3f}   (~{round(rf*50)} of the top 50 right)")
print(f"The learned model is roughly {rf/base:.1f}x the rule on this metric, on this starter slice.")
print("This is the evidence gap Lane 2 is built to investigate properly -- with a future-window")
print("label and honest validation, not just this beginner proxy.")

Working dir: /content/FlyRank-Starter/FlyRank-Starter
Starter data found. You're ready.
Hand-written rule  Precision@50: 0.240   (~12 of the top 50 right)
Random forest      Precision@50: 0.740   (~37 of the top 50 right)
The learned model is roughly 3.1x the rule on this metric, on this starter slice.
This is the evidence gap Lane 2 is built to investigate properly -- with a future-window
label and honest validation, not just this beginner proxy.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Search question:** Given a client's inventory of content pages, which pages should a content editor review first this week for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis:** one row = one pseudonymized content item (`content_id`), described by its trailing-90-day metrics (impressions, clicks, sessions, CTR, position, engagement, freshness, word count). Not a client, not a day — a page.

**Output:** a ranked review queue — top-N pages by opportunity score, each with a reason code (e.g. `stale_visible_page`, `declining_with_demand`, `page_one_decay_risk`) and a suggested action category (refresh / expand / protect / prune / monitor).

**Who acts, and how:** a content/SEO editor with limited weekly review capacity (say, 20-50 pages) pulls from the top of the queue instead of reviewing pages in an arbitrary order (newest-first, alphabetical, or gut feel).

**Cost of a wrong call:**
- *False positive* (queue says "review this," but the page was actually fine): wastes an editor's scarce hour on a page that didn't need it — that hour is now not spent on a page that did.
- *False negative* (a genuinely declining, high-demand page never surfaces): the page keeps losing visibility/traffic until the next review cycle catches it, if it ever does. Given that, in the starter data, 43.8% of pages are already both declining and have real demand (`impressions_90d >= 100`), missing the wrong ones compounds — this isn't a rare-event problem where false negatives are cheap.

Because editor time is the scarce resource on one side and traffic decay compounds on the other, the ranking's *ordering* at the top of the list matters more than raw accuracy across all 30,000 rows — which is why I'll lean on precision@K rather than plain accuracy.

**Why data/ML helps at all:** a single if-statement rule *is* a reasonable first pass (that's exactly what the starter baseline is), but the signals that actually predict "worth reviewing" are tangled — freshness, position, CTR, volume, and engagement interact and trade off against each other in ways that shift by content type and client. The starter results (baseline precision@50 0.240 vs. random forest 0.740) suggest there's real, learnable structure a hand-written rule doesn't capture.

In [19]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
weekly_capacity = 50  # a realistic weekly review capacity for one editor

print(f"Pages already declining AND showing real demand (>=100 impressions/90d): "
      f"{len(declining_with_demand):,} of {len(df):,} ({len(declining_with_demand)/len(df):.1%})")
print(f"An editor reviewing {weekly_capacity} pages/week reaches only "
      f"{weekly_capacity/len(declining_with_demand):.1%} of that pool in a single week.")
print("That gap is why ORDER matters more than just finding candidates -- a plain list isn't")
print("enough; a false negative near the top of a ranked queue is far costlier than one near")
print("the bottom, since the bottom may never get reached at all.")


Pages already declining AND showing real demand (>=100 impressions/90d): 13,152 of 30,000 (43.8%)
An editor reviewing 50 pages/week reaches only 0.4% of that pool in a single week.
That gap is why ORDER matters more than just finding candidates -- a plain list isn't
enough; a false negative near the top of a ranked queue is far costlier than one near
the bottom, since the bottom may never get reached at all.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

The starter CSV has 30,000 rows across 32 clients — one row per pseudonymized content item, trailing-90-day metrics only. Three numbers from the cell below make the case for Lane 2:

- **43.8%** of pages count as `declining_with_demand` (trend down, and still getting 100+ impressions in 90 days) — a large enough pool that a ranking problem is real, not a rounding error, and small enough that an editor genuinely needs a priority order rather than reviewing all of them.
- **23.6%** of pages are `page_one_decay_risk` (currently ranking top-10, and old enough — 180+ days — that decay is a live risk) — these are the highest-value pages to protect, since they already earn visibility.
- **0.06%** (17 rows) are `stale_visible_page` under the strict rule (no update in 180+ days AND 500+ impressions) — a reminder that a single hand-written rule can be *too* strict and miss most of the real opportunity, which is itself evidence for why a scored ranking (not one yes/no rule) is worth building.

In [20]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

total_rows = len(df)
n_clients = df["client_id"].nunique()

# Reason-code style filters straight from the lane guide (section 5), computed on THIS data
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
page_one_decay_risk = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)]
stale_visible_page = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]

print(f"rows: {total_rows:,} across {n_clients} clients")
print(f"declining_with_demand (trend down AND impressions_90d>=100): "
      f"{len(declining_with_demand):,} rows ({len(declining_with_demand)/total_rows:.1%})")
print(f"page_one_decay_risk (top-10 position AND age>=180d):        "
      f"{len(page_one_decay_risk):,} rows ({len(page_one_decay_risk)/total_rows:.1%})")
print(f"stale_visible_page (no update in 180d+ AND impressions>=500): "
      f"{len(stale_visible_page):,} rows ({len(stale_visible_page)/total_rows:.2%})")

rows: 30,000 across 32 clients
declining_with_demand (trend down AND impressions_90d>=100): 13,152 rows (43.8%)
page_one_decay_risk (top-10 position AND age>=180d):        7,076 rows (23.6%)
stale_visible_page (no update in 180d+ AND impressions>=500): 17 rows (0.06%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim by the end of this project:**
- *Observed* associations between content/search signals (freshness, position, CTR, volume, engagement) and a defined outcome (decline or recovery over a future window I will specify).
- *Directional* evidence — e.g. "pages with X pattern were more likely to keep declining" — with an honest effect size, not just "there is a link."
- *Decision-support* output: a ranked queue with reason codes an editor can inspect and override. The queue orders pages by evidence, it does not promise any individual page will improve.

**What I will never claim:**
- That refreshing a page *causes* recovery — I have no experiment or causal design, only observational data, so at most I can say a pattern is associated with later recovery.
- Anything about Google's ranking algorithm, or that I've reverse-engineered it.
- That the current starter label (`trend_direction == "down"`) is the real target — it's a same-window proxy bucket, not a future outcome, so results built on it are a beginner baseline, not the capstone claim. My actual target will be a future-window label (prior 90 days of features -> next 30 days outcome), audited for leakage before I trust it.
- That a result validated on this 30,000-row starter slice generalizes to the ~79M-row warehouse without being re-earned there, with proper client-holdout or time-aware validation.

In [21]:
print(df["trend_direction"].value_counts())
print()
print("trend_pct / trend_direction are proxy fields used only to build a future label later,")
print("never as model features -- per the label trap in skills/flyrank/flyrank-data/SKILL.md.")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct / trend_direction are proxy fields used only to build a future label later,
never as model features -- per the label trap in skills/flyrank/flyrank-data/SKILL.md.


## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.